# Process all data from the paper
1. Basic checks: directionality
2. Preprocess and subset the data into separate csv files, one for each protein
3. Prep the data for MSA (colabfold and EVCouplings)
4. Visualize the data distribution and find the DMS binarization threshold
5. Map the Uniprot ID to SwissProt ID and save each protein as `DMS_ID.csv`
6. Fill out the [DMS_substitution_internal](https://docs.google.com/spreadsheets/d/1M1dKN0wpvE2sKDt4hwrIsTBS1r1HbBObUxjO4LNOQ64/edit?gid=560024392#gid=560024392) sheet for all domains

In [1]:
import pandas as pd
import csv
import subprocess
import sys

## Load the dataset
\* means stop codon

In [2]:
domainome = pd.read_csv('/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/DMS_assays/raw_data/Beltran_Lehner_2025/SupplementaryTable2.txt', 
                        sep='\t')

In [3]:
domainome.head(10)

,domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank
0,A0A2R8Y422_PF00240_2,A0A2R8Y422,*IFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,*,True,118.0,113.0,62.0,10.0,29.0,2.0,97.66667,0.030945,0.014885,-0.819050,0.208478,339
1,A0A2R8Y422_PF00240_2,A0A2R8Y422,AIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,A,False,219.0,277.0,217.0,86.0,225.0,137.0,237.66670,0.069376,0.006673,-0.280790,0.093461,339
2,A0A2R8Y422_PF00240_2,A0A2R8Y422,CIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,C,False,706.0,726.0,459.0,768.0,507.0,616.0,630.33330,0.082052,0.004141,-0.103250,0.057995,339
3,A0A2R8Y422_PF00240_2,A0A2R8Y422,DIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,D,False,407.0,431.0,323.0,508.0,159.0,111.0,387.00000,0.071003,0.005162,-0.258003,0.072296,339
4,A0A2R8Y422_PF00240_2,A0A2R8Y422,EIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,E,False,37.0,56.0,37.0,201.0,102.0,95.0,43.33333,0.116326,0.012085,0.376783,0.169263,339
5,A0A2R8Y422_PF00240_2,A0A2R8Y422,FIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,F,False,264.0,228.0,142.0,334.0,564.0,472.0,211.33330,0.100066,0.006000,0.149047,0.084030,339
6,A0A2R8Y422_PF00240_2,A0A2R8Y422,GIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,G,False,1006.0,1062.0,867.0,852.0,690.0,773.0,978.33330,0.076036,0.003682,-0.187512,0.051568,339
7,A0A2R8Y422_PF00240_2,A0A2R8Y422,HIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,H,False,612.0,695.0,510.0,922.0,326.0,517.0,605.66670,0.081447,0.004220,-0.111728,0.059101,339
8,A0A2R8Y422_PF00240_2,A0A2R8Y422,IIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,I,False,351.0,378.0,228.0,916.0,755.0,883.0,319.00000,0.109426,0.004950,0.280147,0.069333,339
9,A0A2R8Y422_PF00240_2,A0A2R8Y422,KIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,K,False,84.0,85.0,47.0,175.0,236.0,38.0,72.00000,0.102229,0.009503,0.179340,0.133105,339


In [3]:
domainome[domainome['wt_aa'] == domainome['mut_aa']]['normalized_fitness'].value_counts()

Series([], Name: normalized_fitness, dtype: int64)

No synonymous variants

## Check on the directionality of the `normalized_fitness` column:
1. check the value: if negative, then -1 is less fit, though the numbers aren't normalized to 1. The mean/median is higher when it's not a stop codon. So the directionality is correct
2. check the experiments
- Fig 1a and b, `variants that cause unfolding/degradation of the target domain result in a decrease in concentration in the cell, leading to impaired cell growth`
- the data table (above), for those with a stop codon, the input values (read counts usually) are higher than the output values
- Fig 2, there is a heatmap bar for normalized fitness

In [4]:
domainome[domainome['mut_aa'] == "*"]['normalized_fitness'].describe()

count    26848.000000
mean        -1.082122
std          0.439326
min         -5.297542
25%         -1.221094
50%         -1.007839
75%         -0.847451
max          1.475415
Name: normalized_fitness, dtype: float64

In [5]:
domainome[domainome['mut_aa'] != "*"]['normalized_fitness'].describe()

count    536686.000000
mean         -0.280223
std           0.340482
min          -4.122925
25%          -0.507694
50%          -0.187892
75%          -0.032637
max           2.052570
Name: normalized_fitness, dtype: float64

## Data preprocessing:
1. remove rows where the `position` is NA or where the `normalized_fitness` is NA
2. cast `position`, input/output counts as `int`
3. remove all the stop codons
4. subset the dataframe based on unique domain_IDs because multiple domains can map to the same Uniprot ID

In [7]:
domainome.dtypes

domain_ID                    object
uniprot_ID                   object
aa_seq                       object
wt_aa                        object
position                    float64
mut_aa                       object
STOP                         object
input_count_rep1            float64
input_count_rep2            float64
input_count_rep3            float64
output_count_rep1           float64
output_count_rep2           float64
output_count_rep3           float64
mean_input_count            float64
fitness                     float64
fitness_sigma               float64
normalized_fitness          float64
normalized_fitness_sigma    float64
quality_rank                  int64
dtype: object

In [3]:
domainome = domainome[(domainome['position'].notnull()) & (domainome['normalized_fitness'].notnull())]
dtype={'position': int, 
       'input_count_rep1': int, 
       'input_count_rep2': int, 
       'input_count_rep3': int, 
       'output_count_rep1': int,
       'output_count_rep2': int,
       'output_count_rep3': int}
domainome = domainome.astype(dtype)

In [5]:
domainome['domain_ID'].nunique()

522

In [6]:
domainome['uniprot_ID'].nunique()

434

In [4]:
domain_id_list = list(domainome['domain_ID'].drop_duplicates())
idmapping_csv = '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/DMS_assays/processed_data/Beltran_Lehner_2025/idmapping_2025_06_10.csv'
with open(idmapping_csv, mode='r') as f_in:
    reader = csv.reader(f_in)
    idmapping_dict = {rows[0]:rows[1] for rows in reader}

In [5]:
def make_dms_id(domain_id):
    protein, domain, idx = domain_id.split('_')
    if protein in idmapping_dict:
        dms_id = "_".join([idmapping_dict[protein], "Beltran_2025", domain, idx])
    else:
        dms_id = "_".join([domain_id, "Beltran_2025"])
    return dms_id

In [7]:
for domain_id in domain_id_list:
    df = domainome[(domainome["domain_ID"] == domain_id) & (domainome['mut_aa'] != "*")]
    dms_id = make_dms_id(domain_id)
    df.to_csv(f"/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/DMS_assays/processed_data/Beltran_Lehner_2025/{dms_id}.csv", index=False)

### TO-DO
- ~rename the csv files based on the id mapping below~
- clean up the .tmp files

For PRIMO project inclusion
- Double check how many mutations for each domain? Looks like singles only? Read the paper to check
- Are there indels? **No**
- Figure out how they calculate all the scores
- **Is normalized fitness direction correct? 1 being more fit, -1 being less fit**
- **Do we want to filter out the stop codon mutants?**
  Leaning toward yes, because stop codons can lead to indel mutants (or large deletions), these are removed in PG1.
  Otherwise, we have to research how to better represent the stop codons for different models

**TO DO** fill in the data description here (copied from the PG2 meeting blog)

Next step:

- Compute MSAs and PDBs (AF2 predictions of the structure) for 500+ domains - look at domain-specific assay
- Use ColabFold alignments - for EVE, can miss chunks in PTEN and the number of sequences is a lot lower (<1 day)
- Or if use EVCouplings (short sequences with low bitscores will go OOM), it may take longer
- Some papers are reporting better (GEM, PoET) performance with ColabFold MSAs (no need to filter out distant sequences, Aaron added a column of coverage filter) than EVCouplings MSAs (need to filter out distant sequences) - JackHammer tends to retain these distant sequences, only helpful for alignment-based models
- Hard to standardize MSAs for different models
- Need to decide which MSA we want going forward
- AF2 will be based on the same MSA (AF2 predictions give structures and MSAs)

## MSA data prep (colabfold and EVCouplings)
1. To run MSA using colabfold: Generate the wild-type domain sequence and compare with [the original sequence generated by Aaron](/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/ColabFold/Beltran_Lehner_2025/beltran_domains.fa)

In [9]:
with open('/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/ColabFold/Beltran_Lehner_2025/beltran_domains_filtered.fa', 'w') as f_out:
    for domain_id in domain_id_list:
        df = domainome[domainome["domain_ID"] == domain_id]
        fasta_id = domain_id
        offset = int(domain_id.split("_")[-1])
        aa_seq_list = list(df['aa_seq'].iloc[0])
        # print(len(aa_seq_list))
        position = df['position'].iloc[0] - offset
        # print(position)
        wt_aa = df['wt_aa'].iloc[0]
        aa_seq_list[position] = wt_aa
        aa_seq = "".join(aa_seq_list)
        f_out.write(f">{fasta_id}\n{aa_seq}\n")
        continue

To score AIDO (one of the baselines), generate a customized fasta file

In [6]:
out_dir = '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/DMS_assays/processed_data/Beltran_Lehner_2025/AIDO_fasta/'
for domain_id in domain_id_list:
    df = domainome[domainome["domain_ID"] == domain_id]
    seq_len = len(df['aa_seq'].iloc[0])
    dms_id = make_dms_id(domain_id)

    offset = int(domain_id.split("_")[-1])
    aa_seq_list = list(df['aa_seq'].iloc[0])
    # print(len(aa_seq_list))
    position = df['position'].iloc[0] - offset
    # print(position)
    wt_aa = df['wt_aa'].iloc[0]
    aa_seq_list[position] = wt_aa
    aa_seq = "".join(aa_seq_list)
    
    with open(out_dir + dms_id + '.fasta', 'w') as f_out:
        f_out.write(f">{dms_id} 1-{seq_len}\n{aa_seq}\n")
    continue

In [13]:
df = pd.read_csv('/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/Beltran_Lehner_2025/temp_ref_file.csv')
df.columns
# list(df['DMS_id'])

Index(['DMS_id', 'MSA_bitscore', 'MSA_num_seqs', 'MSA_perc_cov', 'MSA_num_cov',
       'MSA_N_eff', 'MSA_Neff_L', 'MSA_Neff_L_category', 'weight_file_name',
       'MSA_filename', 'EVCouplings_model_filename', 'target_aa_seq',
       'DMS_filename', 'offset', 'seq_len', 'pdb_file', 'UniProt_ID',
       'MSA_start', 'MSA_end', 'MSA_len', 'MSA_theta', 'taxon',
       'source_organism', 'includes_multiple_mutants', 'DMS_wildtype_score',
       'first_author', 'coarse_selection_type'],
      dtype='object')

In [10]:
!diff /n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/ColabFold/Beltran_Lehner_2025/beltran_domains_filtered.fa /n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/ColabFold/Beltran_Lehner_2025/beltran_domains.fa

1044a1045
> 


Get the target sequence into a .csv for model scoring

In [9]:
with open('/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/Beltran_Lehner_2025/target_seq.csv', 'w') as f_out:
    f_out.write("DMS_id,target_seq\n")
    for domain_id in domain_id_list:
        df = domainome[domainome["domain_ID"] == domain_id]
        dms_id = make_dms_id(domain_id)
        offset = int(domain_id.split("_")[-1])
        aa_seq_list = list(df['aa_seq'].iloc[0])
        position = df['position'].iloc[0] - offset
        wt_aa = df['wt_aa'].iloc[0]
        aa_seq_list[position] = wt_aa
        aa_seq = "".join(aa_seq_list)
        f_out.write(f"{dms_id},{aa_seq}\n")
        continue

#### Summary of colabfold data prep: 
Check the sequences with Aaron's [on AWS](/home/ubuntu/beltran_domains) - **the sequences are the same**

2. Generate the file to examine number of unique domains, as requested by Debbie on March 31

Column names: UniProt ID, PFID, AA sequence, length, the number of sequences

In [8]:
domainome.head(10)

,domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank
0,A0A2R8Y422_PF00240_2,A0A2R8Y422,*IFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2,*,True,118,113,62,10,29,2,97.66667,0.030945,0.014885,-0.819050,0.208478,339
1,A0A2R8Y422_PF00240_2,A0A2R8Y422,AIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2,A,False,219,277,217,86,225,137,237.66670,0.069376,0.006673,-0.280790,0.093461,339
2,A0A2R8Y422_PF00240_2,A0A2R8Y422,CIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2,C,False,706,726,459,768,507,616,630.33330,0.082052,0.004141,-0.103250,0.057995,339
3,A0A2R8Y422_PF00240_2,A0A2R8Y422,DIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2,D,False,407,431,323,508,159,111,387.00000,0.071003,0.005162,-0.258003,0.072296,339
4,A0A2R8Y422_PF00240_2,A0A2R8Y422,EIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2,E,False,37,56,37,201,102,95,43.33333,0.116326,0.012085,0.376783,0.169263,339
5,A0A2R8Y422_PF00240_2,A0A2R8Y422,FIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2,F,False,264,228,142,334,564,472,211.33330,0.100066,0.006000,0.149047,0.084030,339
6,A0A2R8Y422_PF00240_2,A0A2R8Y422,GIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2,G,False,1006,1062,867,852,690,773,978.33330,0.076036,0.003682,-0.187512,0.051568,339
7,A0A2R8Y422_PF00240_2,A0A2R8Y422,HIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2,H,False,612,695,510,922,326,517,605.66670,0.081447,0.004220,-0.111728,0.059101,339
8,A0A2R8Y422_PF00240_2,A0A2R8Y422,IIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2,I,False,351,378,228,916,755,883,319.00000,0.109427,0.004950,0.280147,0.069333,339
9,A0A2R8Y422_PF00240_2,A0A2R8Y422,KIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2,K,False,84,85,47,175,236,38,72.00000,0.102229,0.009503,0.179340,0.133105,339


Parse uniprot_ID and PFID and wild type AA sequence

In [5]:
with open('/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/DMS_assays/processed_data/Beltran_Lehner_2025/summary.tmp', 'w') as f_out:
    for domain_id in domain_id_list:
        df = domainome[domainome["domain_ID"] == domain_id]
        uniprot_id, pfid, _ = domain_id.split('_')
        offset = int(domain_id.split("_")[-1])
        aa_seq_list = list(df['aa_seq'].iloc[0])
        position = df['position'].iloc[0] - offset
        wt_aa = df['wt_aa'].iloc[0]
        aa_seq_list[position] = wt_aa
        aa_seq = "".join(aa_seq_list)
        aa_len = len(aa_seq)
        start_idx = offset
        end_idx = offset + aa_len - 1
        domain_list = f'({pfid}, {start_idx}, {end_idx})'
        f_out.write("\t".join([uniprot_id, aa_seq, domain_list])+'\n')
        continue

Count the number of sequences under a domain (usually in different uniprot ID)

In [32]:
df = pd.read_csv('/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/DMS_assays/processed_data/Beltran_Lehner_2025/summary.tmp', 
                 sep="\t", names=['uniprot_ID', 'wt_aa_seq', 'domain_list'])
df = df.sort_values('PFID')
df.to_csv('/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/DMS_assays/processed_data/Beltran_Lehner_2025/summary.tsv')
# clean_df = df.groupby(["PFID", 'uniprot_ID']).count()
# clean_df.to_csv('/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/DMS_assays/processed_data/Beltran_Lehner_2025/summary.tmp.sorted', sep='\t')
# clean_df.drop_duplicates(['wt_aa_seq', 'wt_aa_seq_len']).to_csv('/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/DMS_assays/processed_data/Beltran_Lehner_2025/summary.tmp.sorted', sep='\t')
# df.groupby(["PFID", 'wt_aa_seq']).count()


uniprot_ID  \
PFID     wt_aa_seq                                                        
PF00010  RSTHNEMEKNRRAHLRLCLEKLKGLVPLGPESSRHTTLSLLTKAKLH...           1   
         STHNELEKNRRAHLRLCLERLKVLIPLGPDCTRHTTLGLLNKAKAHI...           1   
PF00013  GNAVQEIMIPASKAGLVIGKGGETIKQLQERAGVKMVMIQDGPQNTG...           1   
         GPIITTQVTIPKDLAGSIIGKGGQRIKQIRHESGASIKIDEPLEGSE...           1   
         GQTTVQVRVPYRVVGLVVGPKGATIKRIQQQTHTYIVTPSRDKEPVF...           1   
...                                                                 ...   
PF18410  GCWSIEHVEQYLGTDELPKNDLITYLQKNADAAFLRHWKLTGTNKSI...           1   
PF18694  IRVTEDENDEPIEIPSEDDGTVLLSTVTAQFPGACGLRYRNPVSQCM...           1   
rockdoms QETIEVEDEEEARRVAKELRKKGYEVKIERRGNKWHVHRT                     1   
         RKWEEIAERLREEFNINPEEAREAVEKAGGNEEEARRIVKKRL                  1   
         TTIKVNGQEYTVPLSPEQAAKAAKKRWPDYEVQIHGNTVKVTR                  1   

                                                             wt_aa_seq_len  
PFID     wt_aa_seq                                                          
PF00010  RSTHNEMEKNRRAHLRLCLEKLKGLVPLGPESSRHTTLSLLTKAKLH...              1  
         STHNELEKNRRAHLRLCLERLKVLIPLGPDCTRHTTLGLLNKAKAHI...              1  
PF00013  GNAVQEIMIPASKAGLVIGKGGETIKQLQERAGVKMVMIQDGPQNTG...              1  
         GPIITTQVTIPKDLAGSIIGKGGQRIKQIRHESGASIKIDEPLEGSE...              1  
         GQTTVQVRVPYRVVGLVVGPKGATIKRIQQQTHTYIVTPSRDKEPVF...              1  
...                                                                    ...  
PF18410  GCWSIEHVEQYLGTDELPKNDLITYLQKNADAAFLRHWKLTGTNKSI...              1  
PF18694  IRVTEDENDEPIEIPSEDDGTVLLSTVTAQFPGACGLRYRNPVSQCM...              1  
rockdoms QETIEVEDEEEARRVAKELRKKGYEVKIERRGNKWHVHRT                        1  
         RKWEEIAERLREEFNINPEEAREAVEKAGGNEEEARRIVKKRL                     1  
         TTIKVNGQEYTVPLSPEQAAKAAKKRWPDYEVQIHGNTVKVTR                     1  

[522 rows x 2 columns]

In [31]:
df['PFID'].nunique()

127

Next steps: get the list of domains and start/stop indices for each target protein

3. Generate the .csv file required for EVCouplings run

In [5]:
with open('/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/input/Beltran_Lehner_2025/sequences.csv', 'w') as f_out:
    f_out.write(",".join(['UniProt_ID','target_seq','MSA_region','MSA_theta']) + '\n')
    for domain_id in domain_id_list:
        df = domainome[domainome["domain_ID"] == domain_id]
        fasta_id = domain_id
        offset = int(domain_id.split("_")[-1])
        aa_seq_list = list(df['aa_seq'].iloc[0])
        # print(len(aa_seq_list))
        position = df['position'].iloc[0] - offset
        # print(position)
        wt_aa = df['wt_aa'].iloc[0]
        aa_seq_list[position] = wt_aa
        aa_seq = "".join(aa_seq_list)
        f_out.write(f"{fasta_id},{aa_seq},,0.2\n")
        continue

## Visualize the data and find the DMS_binarization_threshold

In [ ]:
import requests
import json
def get_pfam_domains_for_proteins(protein_sequences):
 """Searches for Pfam domains in a dictionary of protein sequences."""
 pfam_domains = {}
 for protein_id, sequence in protein_sequences.items():
     try:
         # Construct the API request URL
         url = f"https://www.ebi.ac.uk/interpro/api/entry/pfam/PF02171/protein/UniProt/{protein_id}"  # Replace PF02171 with the Pfam entry you want to search for
         response = requests.get(url)
         response.raise_for_status()
         data = response.json()
         # Extract the relevant information from the JSON response
         pfam_domains[protein_id] = data
     except requests.exceptions.RequestException as e:
         print(f"Error searching for Pfam domains for {protein_id}: {e}")
 return pfam_domains

## Map the Uniprot ID to SwissProt ID and save each protein as `DMS_ID.csv` (move this to the very front)

Use [this](uniprot.org/id-mapping) mapping website

Write the Uniprot ID to a csv for submission

In [10]:
pd.Series(domainome['uniprot_ID'].unique()).to_csv('/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/DMS_assays/raw_data/Beltran_Lehner_2025/uniprot_id.csv', index=False, header=False)

Load the mapping result (total 434 proteins)
- 3 rocklin domain proteins have no Swiss-Prot (expected)
- 3 others have entries deleted, manually input the most recent Swiss-Prot ID or available TrEMBL ID

In [2]:
mapping_df = pd.read_csv('/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/DMS_assays/raw_data/Beltran_Lehner_2025/idmapping_2025_06_10.tsv', sep="\t")
mapping_df

,From,Entry,Entry Name,Protein names,Organism,Length,Reviewed
0,A0PJY2,A0PJY2,FEZF1_HUMAN,Fez family zinc finger protein 1 (Zinc finger ...,Homo sapiens (Human),475,reviewed
1,A1X283,A1X283,SPD2B_HUMAN,SH3 and PX domain-containing protein 2B (Adapt...,Homo sapiens (Human),911,reviewed
2,A6NK59,A6NK59,ASB14_HUMAN,Ankyrin repeat and SOCS box protein 14 (ASB-14),Homo sapiens (Human),587,reviewed
3,O00308,O00308,WWP2_HUMAN,NEDD4-like E3 ubiquitin-protein ligase WWP2 (E...,Homo sapiens (Human),870,reviewed
4,O00330,O00330,ODPX_HUMAN,"Pyruvate dehydrogenase protein X component, mi...",Homo sapiens (Human),501,reviewed
...,...,...,...,...,...,...,...
423,Q9Y4J8,Q9Y4J8,DTNA_HUMAN,Dystrobrevin alpha (DTN-A) (Alpha-dystrobrevin...,Homo sapiens (Human),743,reviewed
424,Q9Y5K6,Q9Y5K6,CD2AP_HUMAN,CD2-associated protein (Adapter protein CMS) (...,Homo sapiens (Human),639,reviewed
425,Q9Y618,Q9Y618,NCOR2_HUMAN,Nuclear receptor corepressor 2 (N-CoR2) (CTG r...,Homo sapiens (Human),2514,reviewed
426,Q9Y6N9,Q9Y6N9,USH1C_HUMAN,Harmonin (Antigen NY-CO-38/NY-CO-37) (Autoimmu...,Homo sapiens (Human),552,reviewed


In [3]:
idmapping_dict = mapping_df.set_index('Entry')['Entry Name'].to_dict()

{'A0PJY2': 'FEZF1_HUMAN',
 'A1X283': 'SPD2B_HUMAN',
 'A6NK59': 'ASB14_HUMAN',
 'O00308': 'WWP2_HUMAN',
 'O00330': 'ODPX_HUMAN',
 'O14529': 'CUX2_HUMAN',
 'O14640': 'DVL1_HUMAN',
 'O14776': 'TCRG1_HUMAN',
 'O14813': 'PHX2A_HUMAN',
 'O14901': 'KLF11_HUMAN',
 'O14936': 'CSKP_HUMAN',
 'O15151': 'MDM4_HUMAN',
 'O15259': 'NPHP1_HUMAN',
 'O15265': 'ATX7_HUMAN',
 'O15266': 'SHOX_HUMAN',
 'O15344': 'TRI18_HUMAN',
 'O15350': 'P73_HUMAN',
 'O15405': 'TOX3_HUMAN',
 'O15541': 'R113A_HUMAN',
 'O43167': 'ZBT24_HUMAN',
 'O43186': 'CRX_HUMAN',
 'O43295': 'SRGP3_HUMAN',
 'O43353': 'RIPK2_HUMAN',
 'O43374': 'RASL2_HUMAN',
 'O43395': 'PRPF3_HUMAN',
 'O43586': 'PPIP1_HUMAN',
 'O60281': 'ZN292_HUMAN',
 'O60293': 'ZC3H1_HUMAN',
 'O60341': 'KDM1A_HUMAN',
 'O60481': 'ZIC3_HUMAN',
 'O60885': 'BRD4_HUMAN',
 'O75112': 'LDB3_HUMAN',
 'O75360': 'PROP1_HUMAN',
 'O75366': 'AVIL_HUMAN',
 'O75369': 'FLNB_HUMAN',
 'O75382': 'TRIM3_HUMAN',
 'O75400': 'PR40A_HUMAN',
 'O75478': 'TAD2A_HUMAN',
 'O75534': 'CSDE1_HUMAN',
 'O7

In [4]:
idmapping_dict.update({
    'Q3SY89':'ELB3B_HUMAN',
    'A2RRE5':'A2RRE5_HUMAN',
    'A0A2R8Y422': 'A0A2R8Y422_HUMAN'
})

In [5]:
with open('/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/DMS_assays/processed_data/Beltran_Lehner_2025/idmapping_2025_06_10.csv', 'w') as f_out:
    for key, value in idmapping_dict.items():
        f_out.write(f"{key},{value}\n")

## Fill out the [DMS_substitution_internal](https://docs.google.com/spreadsheets/d/1M1dKN0wpvE2sKDt4hwrIsTBS1r1HbBObUxjO4LNOQ64/edit?gid=560024392#gid=560024392) sheet for all domains
- Add a column of the full sequence of the protein and check [DMS_fields](https://docs.google.com/spreadsheets/d/1M1dKN0wpvE2sKDt4hwrIsTBS1r1HbBObUxjO4LNOQ64/edit?gid=560024392#gid=560024392)
- Get the sequence of each domain and add the commas between neighboring domains

Other info
- taxon: Eukaryote
- source_organism: Homo sapiens
- assay_organism: Saccharomyces cerevisiae
- target_aa: parsed below
- target_nt_seq: **need to check**
- full_aa_seq: leave blank
- full_nt_seq: **need to generate from online sources**
- domain_list_target_sequence: **need to convert the current to SwissProt**
- domain_list_full_sequence: **need to generate from online sources**
- include_multiple_mutants: FALSE
- DMS_total_number_mutants: **do we include synonymous for glms?**
- DMS_number_single_mutants: same as above
- DMS_number_multiple_mutants: 0
- DMS_binarization_cutoff: see below
- DMS_wildtype_score: 0
- publication info: omitted, available
- region_mutated: can parse from domain id
- molecule name: **need to generate from online sources**
- selection_assay: Growth
- selection_type: Stability
- library_construction: **?**
- replicates: 3 biological, **need to fill in r^2**
- has_codon_level: FALSE
- MSA_filename: `DMS-ID_b-bitscore.a2m`
- other MSA info: done
- raw_DMS_file_name: SupplementaryTable2.txt
- raw_DMS_phenotype_name: normalized_fitness
- raw_DMS_directionality: 1
- raw_DMS_mutant_column: mut_aa
- raw_DMS_replicate_columns: **no fitness score replicates available, only the counts from 3 replicates**
- weight_file_name: `DMS-ID_theta0.2_b-bitscore.npy`
- pdb_file: `DMS-ID.pdb`
- ProteinGym_version: 1.4
- raw_mut_offset: N/A
- coarse_selection_type: Abundance
- coarse_selection_reason:

In [7]:
def cp_file(from_path, to_path, silent=False):
    try:
        return subprocess.run(
            ["cp", from_path, to_path],
            check=True,
            stdout=subprocess.DEVNULL if silent else None,
            stderr=subprocess.DEVNULL if silent else None,
        )
    except subprocess.CalledProcessError as e:
        raise e

## Create the DMS data from each csv file

In [14]:
sys.path.append("/n/groups/marks/users/elain/PG2/ProteinGym/proteingym")

In [16]:
from utils import data_utils

In [19]:
target_seq_df = pd.read_csv('/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/Beltran_Lehner_2025/target_seq.csv')
target_seq_dict = target_seq_df.set_index('DMS_id')['target_seq'].to_dict()

cleaned_DMS_directory = '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/DMS_assays/processed_data/Beltran_Lehner_2025/cleaned'

In [23]:
domainome.head()

,domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank
0,A0A2R8Y422_PF00240_2,A0A2R8Y422,*IFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2,*,True,118,113,62,10,29,2,97.66667,0.030945,0.014885,-0.819050,0.208478,339
1,A0A2R8Y422_PF00240_2,A0A2R8Y422,AIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2,A,False,219,277,217,86,225,137,237.66670,0.069376,0.006673,-0.280790,0.093461,339
2,A0A2R8Y422_PF00240_2,A0A2R8Y422,CIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2,C,False,706,726,459,768,507,616,630.33330,0.082052,0.004141,-0.103250,0.057995,339
3,A0A2R8Y422_PF00240_2,A0A2R8Y422,DIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2,D,False,407,431,323,508,159,111,387.00000,0.071003,0.005162,-0.258003,0.072296,339
4,A0A2R8Y422_PF00240_2,A0A2R8Y422,EIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2,E,False,37,56,37,201,102,95,43.33333,0.116326,0.012085,0.376783,0.169263,339


In [28]:
for domain_id in domain_id_list:
    dms_id = make_dms_id(domain_id)
    _, _, idx = domain_id.split('_')
    idx = int(idx)
    raw_dms_df = pd.read_csv(f"/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/DMS_assays/processed_data/Beltran_Lehner_2025/{dms_id}.csv")
    raw_dms_df['mutant'] = raw_dms_df.apply(lambda row: row.wt_aa + str(int(row.position) - idx + 1) + row.mut_aa, axis=1)
    raw_dms_df.rename(columns={'aa_seq': 'mutated_sequence', 'normalized_fitness': 'score'}, inplace=True)
    raw_dms_df = raw_dms_df[['mutant', 'mutated_sequence', 'score']]
    raw_dms_df.to_csv(cleaned_DMS_directory + '/' + dms_id + '.tmp', index=False)
    target_seq = target_seq_dict[dms_id]
    cleaned_df = data_utils.DMS_file_cleanup(cleaned_DMS_directory + '/' + dms_id + '.tmp', target_seq)
    cleaned_df.to_csv(cleaned_DMS_directory + '/' + dms_id + '.csv', index=False)

## Move all the pdb files to one directory

In [8]:
destination_directory = '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/pdbs'
from_directory = '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/DMS_AF2/Beltran_Lehner_2025/output'

for domain_id in domain_id_list:
    from_pdb_file = from_directory + '/' + domain_id + '/ranked_0.pdb'
    dms_id = make_dms_id(domain_id)
    to_pdb_file = destination_directory + '/' + dms_id + '.pdb'
    cp_file(from_pdb_file, to_pdb_file)

## Add pdb_range to the reference file

In [2]:
df = pd.read_csv('/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/Beltran_Lehner_2025/temp_ref_file.csv')
df.head()

,DMS_id,MSA_bitscore,MSA_num_seqs,MSA_perc_cov,MSA_num_cov,MSA_N_eff,MSA_Neff_L,MSA_Neff_L_category,weight_file_name,MSA_filename,...,MSA_start,MSA_end,MSA_len,MSA_theta,taxon,source_organism,includes_multiple_mutants,DMS_wildtype_score,first_author,coarse_selection_type
0,A0A2R8Y422_HUMAN_Beltran_2025_PF00240_2,0.7,61559,0.971,68,5929.1,84.701140,medium,A0A2R8Y422_HUMAN_Beltran_2025_PF00240_2_theta_...,A0A2R8Y422_HUMAN_Beltran_2025_PF00240_2_theta_...,...,1,70,70,0.2,Human,Saccharomyces cerevisiae,False,0,Beltran,Abundance
1,FEZF1_HUMAN_Beltran_2025_PF00096_289,1.5,645870,0.955,21,1813.5,82.430408,medium,FEZF1_HUMAN_Beltran_2025_PF00096_289_theta_0.8...,FEZF1_HUMAN_Beltran_2025_PF00096_289_theta_0.8...,...,1,22,22,0.2,Human,Saccharomyces cerevisiae,False,0,Beltran,Abundance
2,SPD2B_HUMAN_Beltran_2025_PF00018_155,0.9,33216,0.945,52,721.9,13.125128,medium,SPD2B_HUMAN_Beltran_2025_PF00018_155_theta_0.8...,SPD2B_HUMAN_Beltran_2025_PF00018_155_theta_0.8...,...,1,55,55,0.2,Human,Saccharomyces cerevisiae,False,0,Beltran,Abundance
3,SPD2B_HUMAN_Beltran_2025_PF00018_222,0.9,45592,0.898,53,947.7,16.062838,medium,SPD2B_HUMAN_Beltran_2025_PF00018_222_theta_0.8...,SPD2B_HUMAN_Beltran_2025_PF00018_222_theta_0.8...,...,1,59,59,0.2,Human,Saccharomyces cerevisiae,False,0,Beltran,Abundance
4,A2RRE5_HUMAN_Beltran_2025_PF01846_267,0.7,4543,0.952,60,128.7,2.042930,medium,A2RRE5_HUMAN_Beltran_2025_PF01846_267_theta_0....,A2RRE5_HUMAN_Beltran_2025_PF01846_267_theta_0....,...,1,63,63,0.2,Human,Saccharomyces cerevisiae,False,0,Beltran,Abundance


In [3]:
df['pdb_range'] = df['MSA_start'].astype(str) + '-' + df['MSA_end'].astype(str)
df.to_csv('/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/Beltran_Lehner_2025/temp_ref_file_pdb_range.csv', index=False)